In [ ]:
%load_ext autoreload
%autoreload 2

# liftover_pos

> Liftover between human genomes for cpg positions.

The `liftover_ps` module provides comprehensive tools for genomic coordinate conversion and CpG site validation. It handles liftover operations across different genome builds and includes validation utilities to ensure data integrity.

In [ ]:
#| default_exp liftover_ps

In [ ]:
#| export
from __future__ import annotations

In [ ]:
#| hide
from nbdev.showdoc import *

/home/magyary/anaconda3/envs/kerepesi_2025/lib/python3.12/site-packages/nbdev/doclinks.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources,importlib


## Coordinate Processing and Validation

This module provides tools for:
- **Genomic Liftover**: Convert coordinates between different genome builds (e.g., hg19 to hg38).
- **CpG Validation**: Fetch sequences at positions and verify they contain valid CpG sites.
- **Reference Detection**: Automatically determine the most likely reference genome for unknown data.

### Import Dependencies

### Working with Sample Data

In [ ]:
# export
import pandas as pd
import re
from bs_cpg.setup import *
from bs_cpg.download_ref import *
from pathlib import Path
import pysam

In [ ]:
# Example: Load sample CpG data
df = read_sample_cpg(["chromosome", "pos"])
print(f"Sample data shape: {df.shape}")
print(df.head())


,chromosome,pos
1594976,chr11,30607691
41581,chr1,10269003
2926242,chrX,55066102
573570,chr3,169781668
1930113,chr14,59950571
...,...,...
2556779,chr19,17438314
452764,chr2,241331303
1554781,chr10,135171502
898587,chr6,31869119


In [ ]:
# View first few chromosome values
print("Sample chromosomes:")
print(df["chromosome"].to_list()[:5])


['chr11', 'chr1', 'chrX', 'chr3', 'chr14']

In [ ]:
# import pysam
# fasta = pysam.FastaFile("data/hg38.fa")
# def fetch_seq(row):
#     return fasta.fetch(row["new_chrom"], row["Start"], row["End"])

In [ ]:
## CpG Site Validation

### Fetching Sequences from Reference Genomes

# Example: Open a reference genome
# fasta = pysam.FastaFile(str(get_ref_genome("hg19")))


✅ Final file '/mnt/idms/home/magyary/.bs-cpg/hg19.fa.bgz' already exists.


In [ ]:
# Example: Fetch a 50bp sequence from chr1 (0-based coordinates)
# seq = fasta.fetch(reference="chr1", start=90000, end=90050)
# print(f"Sequence: {seq}")


'tctcttgctgccctggagaccagctgccccacgaaggaaacagagccaac'

In [ ]:
#| export

from typing import Sequence, Union

def cpg_reads(chromosomes: Sequence, positions: Sequence, ref_genome: 'Path | str | pysam.FastaFile', index_base: int = 1):
    """
    Fetch the 2bp sequence starting at each given genomic position from a reference genome.

    This function retrieves CpG sites from a reference genome by fetching 2-base sequences at
    specified positions. It intelligently handles different chromosome naming conventions (e.g., 'chr1' vs '1').

    Args:
        chromosomes: list-like of chromosome names (e.g., 'chr1' or '1').
        positions: list-like of genomic positions (ints). If index_base==1, positions are 1-based; if 0, 0-based.
        ref_genome: genome name (e.g., 'hg19'), a path to a bgzipped FASTA, or an open pysam.FastaFile.
        index_base: 0 or 1 (default: 1). Specifies if input positions are 0-based or 1-based.

    Returns:
        pd.Series: The fetched 2-bp sequences (uppercased). None when unavailable.
    
    Examples:
        >>> # Fetch CpG reads from sample data
        >>> reads = cpg_reads(df['chromosome'], df['pos'], 'hg19', index_base=1)
        >>> print(reads.head())
    """
    import pysam as _pysam
    import pandas as _pd
    from pathlib import Path as _Path
    from bs_cpg.download_ref import get_ref_genome


    # Normalize inputs
    chroms = list(chromosomes.tolist() if hasattr(chromosomes, 'tolist') else chromosomes)
    poss = list(positions.tolist() if hasattr(positions, 'tolist') else positions)
    
    if len(chroms) != len(poss):
        raise ValueError("chromosomes and positions must be the same length")

    # Determine index base
    base = 1
    if isinstance(index_base, (int, float)) and int(index_base) in (0, 1):
        base = int(index_base)
    elif isinstance(index_base, str) and index_base in ("0", "1"):
        base = int(index_base)

    # Resolve/open FASTA
    close_after = False
    if isinstance(ref_genome, _pysam.FastaFile):
        fasta = ref_genome
    else:
        rg_path = ref_genome
        # If a string/path that exists on disk, use as-is; otherwise treat as genome name
        if isinstance(rg_path, (str, _Path)):
            p = _Path(rg_path)
            if p.exists():
                fasta_path = p
            else:
                fasta_path = get_ref_genome(str(rg_path), verbose=False)
        else:
            fasta_path = get_ref_genome(str(rg_path), verbose=False)
        fasta = _pysam.FastaFile(str(fasta_path))
        close_after = True

    # Precompute available references and resolve aliases ('chr1' <-> '1') once per unique chrom
    try:
        ref_set = set(fasta.references)
    except Exception:
        ref_set = set()
    alias_map = {}
    uniq = {str(c) for c in chroms if c is not None and not (_pd.isna(c) if hasattr(_pd, 'isna') else False)}
    for s in uniq:
        cands = (s, (s[3:] if s.startswith('chr') else 'chr' + s))
        chosen = None
        for cand in cands:
            if cand in ref_set:
                chosen = cand
                break
        alias_map[s] = chosen

    out = []
    for chrom, pos in zip(chroms, poss):
        if chrom is None or _pd.isna(chrom) or pos is None or _pd.isna(pos):
            out.append(None)
            continue
        try:
            p = int(pos)
        except Exception:
            out.append(None)
            continue
        start = p - 1 if base == 1 else p
        if start < 0:
            start = 0
        alias = alias_map.get(str(chrom))
        if alias is None:
            out.append(None)
            continue
        try:
            seq = fasta.fetch(reference=str(alias), start=start, end=start + 2)
            out.append(seq.upper() if seq else None)
        except Exception:
            out.append(None)

    if close_after:
        fasta.close()

    return _pd.Series(out, name="cpg_reads")

0      CG
1      CG
2      CG
3      CG
4      CG
       ..
995    CG
996    CG
997    CG
998    CG
999    CG
Name: cpg_reads, Length: 1000, dtype: object

In [ ]:
#| export

def cpg_percent(reads) -> float:
    """
    Calculate the percentage of valid CpG sites in a dataset.
    
    Returns the percent of entries equal to 'CG' (case-insensitive) in `reads`.
    Accepts any iterable/Series; ignores NA/None values.
    
    Args:
        reads: An iterable (list, Series, etc.) of 2-bp sequences.
    
    Returns:
        float: Percentage of entries that are 'CG' (0-100).
    
    Examples:
        >>> reads = cpg_reads(df['chromosome'], df['pos'], 'hg19')
        >>> percent_cg = cpg_percent(reads)
        >>> print(f"CpG sites: {percent_cg:.1f}%")
    """
    import pandas as _pd
    if reads is None:
        return 0.0
    s = _pd.Series(reads)
    s = s.dropna().astype(str).str.upper()
    if len(s) == 0:
        return 0.0
    return 100.0 * (s == "CG").sum() / len(s)

np.float64(100.0)

In [ ]:
#| export

from typing import Sequence, Union, Dict, Any

def guess_ref_and_index_base(
    chromosomes,
    positions,
    ref_genomes: Sequence[Union[str, 'Path']],
    index_bases: Sequence[int] = (0, 1),
) -> Dict[str, Any]:
    """
    Automatically identify the most likely reference genome and index base.
    
    This function tries combinations of reference genomes and index bases,
    calculating the percentage of valid CpG sites for each combination.
    It returns the combination with the highest CpG percentage as the best guess.
    
    Args:
        chromosomes: Chromosome identifiers (e.g., ['chr1', 'chr2']).
        positions: Genomic positions corresponding to chromosomes.
        ref_genomes: List of genome names or paths to test (e.g., ['hg19', 'hg38']).
        index_bases: List of index bases to test (default: [0, 1]).
    
    Returns:
        dict: Contains:
            - best_genome: Most likely genome name/path
            - best_index_base: Most likely index base (0 or 1)
            - best_percent: Percentage of CpG sites with best parameters
            - results: List of dicts with all tried combinations
    
    Examples:
        >>> result = guess_ref_and_index_base(
        ...     df['chromosome'], 
        ...     df['pos'],
        ...     ref_genomes=['hg19', 'hg38']
        ... )
        >>> print(f"Best genome: {result['best_genome']} ({result['best_percent']:.1f}% CpG)")
    """
    from bs_cpg.download_ref import get_ref_genome
    results = []
    for rg in ref_genomes:
        # Resolve genome path string if a genome name was provided
        rg_path = rg
        if isinstance(rg, str):
            rg_path = get_ref_genome(rg, verbose=False)
        for base in index_bases:
            reads = cpg_reads(chromosomes, positions, rg_path, index_base=base)
            pct = cpg_percent(reads)
            results.append({
                "ref_genome": rg if isinstance(rg, str) else str(rg_path),
                "ref_genome_path": str(rg_path),
                "index_base": int(base),
                "percent_cg": float(pct),
            })
    # Pick max by percent_cg
    if results:
        best = max(results, key=lambda x: x["percent_cg"])
        return {
            "best_genome": best["ref_genome"],
            "best_index_base": best["index_base"],
            "best_percent": best["percent_cg"],
            "results": results,
        }
    return {"best_genome": None, "best_index_base": None, "best_percent": 0.0, "results": []}

{'best_genome': 'hg19',
 'best_index_base': 0,
 'best_percent': 100.0,
 'results': [{'ref_genome': 'hg19',
   'ref_genome_path': '/mnt/idms/home/magyary/.bs-cpg/hg19.fa.bgz',
   'index_base': 0,
   'percent_cg': 100.0},
  {'ref_genome': 'hg19',
   'ref_genome_path': '/mnt/idms/home/magyary/.bs-cpg/hg19.fa.bgz',
   'index_base': 1,
   'percent_cg': 0.0},
  {'ref_genome': 'hg38',
   'ref_genome_path': '/mnt/idms/home/magyary/.bs-cpg/hg38.fa.bgz',
   'index_base': 0,
   'percent_cg': 2.992776057791538},
  {'ref_genome': 'hg38',
   'ref_genome_path': '/mnt/idms/home/magyary/.bs-cpg/hg38.fa.bgz',
   'index_base': 1,
   'percent_cg': 1.4447884416924666}]}

In [ ]:
## Genomic Liftover

Converting coordinates between genome builds (e.g., hg19 → hg38) is essential when working with data from different sources or genome versions.


✅ File '/mnt/idms/home/magyary/.bs-cpg/hg19ToHg38.over.chain.gz' already exists. Skipping.


Path('/mnt/idms/home/magyary/.bs-cpg/hg19ToHg38.over.chain.gz')

In [ ]:
# | export

from typing import Sequence, Optional, Tuple, List, Union
import math

try:
    import pandas as _pd
except Exception:
    _pd = None

from liftover import ChainFile  # fast C++ liftover
from bs_cpg.download_ref import get_liftover_chain  # your helper


def liftover_positions(
    chromosomes: Sequence,
    positions: Sequence,
    genome_from: Optional[str] = None,
    genome_to: Optional[str] = None,
    index_base_from: int = 1,
    index_base_to: int = 1,
    return_df: bool = False,
) -> Union[Tuple[List[Optional[str]], List[Optional[int]]], "_pd.DataFrame"]:
    """
    Convert genomic coordinates between different genome builds or adjust index bases.
    
    This function performs liftover operations when both genome_from and genome_to are specified,
    or simple base conversion when both are None. It handles edge cases like unmapped regions,
    invalid coordinates, and chromosome naming variations (e.g., 'chr1' vs '1').
    
    Args:
        chromosomes: Sequence of chromosome identifiers (e.g., ['chr1', 'chr2']).
        positions: Sequence of genomic positions. Must be same length as chromosomes.
        genome_from: Source genome build (e.g., 'hg19'). Required for liftover.
        genome_to: Target genome build (e.g., 'hg38'). Required for liftover.
        index_base_from: Input indexing system (0 or 1, default: 1 for 1-based).
        index_base_to: Output indexing system (0 or 1, default: 1 for 1-based).
        return_df: If True, return results as a pandas DataFrame; else return tuple of lists.
    
    Returns:
        Tuple[List, List] or pd.DataFrame:
            If return_df=False: (new_chromosomes, new_positions) where unmapped positions are None.
            If return_df=True: DataFrame with detailed input/output columns.
    
    Raises:
        ValueError: If chromosomes and positions have different lengths, or invalid bases.
        ImportError: If pandas is required (return_df=True) but not installed.
    
    Examples:
        >>> # Liftover from hg19 to hg38 (1-based to 1-based)
        >>> new_chroms, new_pos = liftover_positions(
        ...     chromosomes=['chr1', 'chr1'],
        ...     positions=[100001, 200000],
        ...     genome_from='hg19',
        ...     genome_to='hg38',
        ...     index_base_from=1,
        ...     index_base_to=1
        ... )
        >>> print(list(zip(new_chroms, new_pos)))
        >>> 
        >>> # Base conversion only (no liftover)
        >>> new_chroms, new_pos = liftover_positions(
        ...     chromosomes=['chr1'],
        ...     positions=[100],
        ...     index_base_from=0,
        ...     index_base_to=1
        ... )
    """

    if len(chromosomes) != len(positions):
        raise ValueError(
            "Input sequences 'chromosomes' and 'positions' must be the same length."
        )

    # materialize sequences
    chroms = list(
        chromosomes.tolist() if hasattr(chromosomes, "tolist") else chromosomes
    )
    poss = list(positions.tolist() if hasattr(positions, "tolist") else positions)

    # normalize bases
    def _nb(x):
        if x in (0, 1, "0", "1"):
            return int(x)
        raise ValueError(f"Invalid index base '{x}'. Must be 0 or 1.")

    base_from = _nb(index_base_from)
    base_to = _nb(index_base_to)

    # hard-normalize types
    def _norm_chr(c):
        if c is None:
            return None
        s = str(c).strip()
        return s if s else None

    def _norm_pos(p):
        if p is None:
            return None
        if isinstance(p, float) and math.isnan(p):
            return None
        try:
            return int(p)
        except Exception:
            return None

    chroms = [_norm_chr(c) for c in chroms]
    poss = [_norm_pos(p) for p in poss]

    out_chroms: List[Optional[str]] = [None] * len(chroms)
    out_pos: List[Optional[int]] = [None] * len(poss)

    # Mode selection
    is_liftover = genome_from is not None and genome_to is not None
    if not is_liftover and not (genome_from is None and genome_to is None):
        raise ValueError(
            "Specify both 'genome_from' and 'genome_to' for liftover, or both None for base conversion."
        )

    if is_liftover:
        # Always use 0-based lifter to make math explicit.
        chain_path = get_liftover_chain(genome_from, genome_to, verbose=False)
        conv = ChainFile(str(chain_path), one_based=False)

        # helper to generate alias candidates per contig
        def _cands(c: str) -> List[str]:
            out = []
            if c.startswith("chr"):
                out += [c, c[3:]]
            else:
                out += [c, f"chr{c}"]
            if c in {"M", "MT", "chrM", "chrMT"}:
                out += ["M", "MT", "chrM", "chrMT"]
            # dedupe while preserving order
            seen, uniq = set(), []
            for x in out:
                if x not in seen:
                    seen.add(x)
                    uniq.append(x)
            return uniq

        for i, (c, p1) in enumerate(zip(chroms, poss)):
            if c is None or p1 is None:  # missing input
                continue
            # convert input to 0-based for the converter
            p0 = p1 - 1 if base_from == 1 else p1
            if p0 < 0:  # invalid in 0-based
                continue

            res_chrom = None
            res_pos0 = None
            for cand in _cands(c):
                try:
                    res = conv.convert_coordinate(
                        cand, p0
                    )  # [] if unmappable or contig absent
                except TypeError:
                    continue
                if res:
                    rc, rp0, *_ = res[0]
                    res_chrom, res_pos0 = rc, int(rp0)
                    break

            if res_chrom is None:
                continue  # unmapped

            # convert output base explicitly
            final_pos = res_pos0 + (1 if base_to == 1 else 0)
            if final_pos >= base_to:
                out_chroms[i] = res_chrom
                out_pos[i] = final_pos

    else:
        # simple base conversion only
        adj = base_to - base_from
        for i, (c, p) in enumerate(zip(chroms, poss)):
            if c is None or p is None:
                continue
            fp = p + adj
            if fp >= base_to:
                out_chroms[i] = c
                out_pos[i] = fp

    if return_df:
        if _pd is None:
            raise ImportError("pandas is required for `return_df=True`.")
        return _pd.DataFrame(
            {
                "chromosome_in": chroms,
                "pos_in": poss,
                "genome_from": genome_from,
                "genome_to": genome_to,
                "index_base_from": base_from,
                "index_base_to": base_to,
                "chromosome_out": out_chroms,
                "pos_out": out_pos,
            }
        )

    return out_chroms, out_pos

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

# Demo code (won't be exported)
if __name__ == "__main__":
    try:
        # Small smoke demo, requires df from earlier cells
        from pathlib import Path
        import pysam
        # Ensure df exists
        try:
            df
        except NameError:
            from bs_cpg.setup import read_sample_cpg
            df = read_sample_cpg(["chromosome", "pos"])
        reads = cpg_reads(df.chromosome, df.pos, "hg19", index_base=0)
        print("CG %:", cpg_percent(reads))
    except Exception as e:
        # non-fatal during notebook runs
        pass

In [ ]:
# quick invariance checks (smoke tests)
# These won't fail the notebook export but help catch accidental regressions when running interactively.
try:
    # 0->0 should match 1->1 adjusted by conversion on both ends.
    ch0, p0 = liftover_positions(["chr1"], [100000], "hg19", "hg38", index_base_from=0, index_base_to=0)
    ch1, p1 = liftover_positions(["chr1"], [100001], "hg19", "hg38", index_base_from=1, index_base_to=1)
    if ch0[0] is not None and ch1[0] is not None:
        assert ch0[0] == ch1[0]
        assert p0[0] == p1[0]
except Exception:
    # Non-fatal in notebook context
    pass